In [1]:
import os
import keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
print(tf.__version__)

2.10.0


In [4]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [5]:
import gc
tf.keras.backend.clear_session()
gc.collect()

0

In [6]:
# Construct Distiller() class
class Distiller(keras.Model):
    def __init__(self, student, teacher):
        super(Distiller, self).__init__()
        self.teacher = teacher
        self.student = student

    def compile(
        self,
        optimizer,
        metrics,
        student_loss_fn,
        distillation_loss_fn,
        alpha=0.5,
        temperature=2,
    ):
        """Configure the distiller.

        Args:
            optimizer: Keras optimizer for the student weights
            metrics: Keras metrics for evaluation
            student_loss_fn: Loss function of difference between student
                predictions and ground-truth
            distillation_loss_fn: Loss function of difference between soft
                student predictions and soft teacher predictions
            alpha: weight to student_loss_fn and 1-alpha to distillation_loss_fn
            temperature: Temperature for softening probability distributions.
                Larger temperature gives softer distributions.
        """
        super().compile(optimizer=optimizer, metrics=metrics)
        self.student_loss_fn = student_loss_fn
        self.distillation_loss_fn = distillation_loss_fn
        self.alpha = alpha
        self.temperature = temperature

    def compute_loss(
        self, x=None, y=None, y_pred=None, sample_weight=None, allow_empty=False
    ):
        teacher_pred = self.teacher(x, training=False)
        student_loss = self.student_loss_fn(y, y_pred)

        distillation_loss = self.distillation_loss_fn(
            ops.softmax(teacher_pred / self.temperature, axis=1),
            ops.softmax(y_pred / self.temperature, axis=1),
        ) * (self.temperature**2)

        loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss
        return loss

    def call(self, x):
        return self.student(x)

In [7]:
# Define directories
train_dir = "C:/Users/bryan/Desktop/Skin Cancer Detection/Dataset/Annotated/train"
val_dir = "C:/Users/bryan/Desktop/Skin Cancer Detection/Dataset/Annotated/valid"
test_dir = "C:/Users/bryan/Desktop/Skin Cancer Detection/Dataset/Annotated/test"

# Load datasets
batch_size = 64
img_size = (224, 224)  # Change based on your dataset

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=img_size, batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=img_size, batch_size=batch_size
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=img_size, batch_size=batch_size
)

# Normalize images (convert to float and scale)
train_ds = train_ds.map(lambda x, y: (x / 255.0, y))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y))
test_ds = test_ds.map(lambda x, y: (x / 255.0, y))

Found 10941 files belonging to 3 classes.
Found 1039 files belonging to 3 classes.
Found 519 files belonging to 3 classes.


In [8]:
# Load pre-trained teacher (ResNet50)
base_teacher = keras.applications.DenseNet169(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Create a new model with dropout
teacher_input = keras.Input(shape=(224, 224, 3))
x = base_teacher(teacher_input)  # Pass input through pre-trained layers
x = layers.GlobalAveragePooling2D()(x)
teacher_output = layers.Dense(3, activation='softmax')(x)
teacher = keras.Model(inputs=teacher_input, outputs=teacher_output)

# Load pre-trained student (MobileNetV2)
base_student = keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Create a new model with dropout
student_input = keras.Input(shape=(224, 224, 3))
y = base_student(student_input)  # Pass input through pre-trained layers
y = layers.GlobalAveragePooling2D()(y)
student_output = layers.Dense(3, activation='softmax')(y)
student = keras.Model(inputs=student_input, outputs=student_output)

In [9]:
# Load pre-trained MobileNetV2 without top layer
base_student = keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze all BatchNorm layers in the student
for layer in base_student.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Build the custom top of the student model
student_input = keras.Input(shape=(224, 224, 3))
y = base_student(student_input)
y = layers.GlobalAveragePooling2D()(y)
student_output = layers.Dense(3, activation='softmax')(y)
student = keras.Model(inputs=student_input, outputs=student_output)

In [ ]:
# from keras.callbacks import EarlyStopping

# early_stopping = EarlyStopping(
#     monitor='val_loss',  # Monitor validation loss
#     patience=5,          # Stop after 5 epochs with no improvement
#     restore_best_weights=True  # Restore weights from the best epoch
# )

In [10]:
# Train teacher as usual
teacher.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.001, momentum=0.9),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)


# Train and evaluate teacher on data.
teacher.fit(train_ds, validation_data=val_ds, epochs=5)#, callbacks=[early_stopping])
teacher.evaluate(test_ds)

Epoch 1/5


c:\Users\bryan\anaconda3\envs\VirPy\lib\site-packages\keras\backend.py:5582: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


ResourceExhaustedError: Graph execution error:

Detected at node 'model/densenet169/conv3_block10_0_bn/FusedBatchNormV3' defined at (most recent call last):
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "c:\Users\bryan\anaconda3\envs\VirPy\lib\site-packages\tornado\platform\asyncio.py", line 205, in start
      self.asyncio_loop.run_forever()
    File "c:\U

In [ ]:
# Initialize and compile distiller
distiller = Distiller(student=student, teacher=teacher)
distiller.compile(
    optimizer=keras.optimizers.Adam(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.5,
    temperature=2,
)

# Distill teacher to student
distiller.fit(train_ds, validation_data=val_ds, epochs=5)#, callbacks=[early_stopping])

# Evaluate student on test dataset
distiller.evaluate(test_ds)

In [ ]:
#Clone student for later comparison
student_scratch = keras.models.clone_model(student)

# Train student as done usually
student_scratch.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

# Train and evaluate student trained from scratch.
student_scratch.fit(train_ds, validation_data=val_ds, epochs=6])#, callbacks=[early_stopping])
student_scratch.evaluate(test_ds)

In [ ]:
# Evaluate the student_scratch model on the test dataset
results = student_scratch.evaluate(test_ds, verbose=0)

# Get the metric names from the model
metric_names = student_scratch.metrics_names

# Print the metric scores
for metric_name, score in zip(metric_names, results):
  print(f'{metric_name}: {score}')

In [ ]:
import matplotlib.pyplot as plt

# Assuming you have the training history for teacher, distiller, and student_scratch
# in variables like teacher_history, distiller_history, and student_scratch_history.
# These history objects should have 'loss' and 'sparse_categorical_accuracy' (or similar) attributes.

# Example (replace with your actual history objects)
teacher_history = teacher.history
distiller_history = distiller.history
student_scratch_history = student_scratch.history

# Plot training & validation accuracy values
plt.figure(figsize=(10, 5))
plt.plot(teacher_history['sparse_categorical_accuracy'])
plt.plot(distiller_history['sparse_categorical_accuracy'])
plt.plot(student_scratch_history['sparse_categorical_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Teacher', 'Distillation', 'Student'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(10, 5))
plt.plot(teacher_history['loss'])
plt.plot(distiller_history['loss'])
plt.plot(student_scratch_history['loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Teacher', 'Distillation', 'Student'], loc='upper left')
plt.show()